# Sakila on Cloud SQL: connect from Python and query

Credentials come from environment variables (`INSTANCE_CONNECTION_NAME`, `MYSQL_ROOT_PW`); nothing secret is in the notebook.

In [1]:
import os
import pandas as pd
from google.cloud.sql.connector import Connector

# Connect to Cloud SQL; credentials come from environment variables
connector = Connector()
cnx = connector.connect(
    os.environ["INSTANCE_CONNECTION_NAME"], 
    "pymysql",
    user="root",
    password=os.environ["MYSQL_ROOT_PW"],
    db="sakila",
)
cursor = cnx.cursor()

# Helper: run a query and show the rows as a table
def run(sql):
    cursor.execute(sql)
    cols = [d[0] for d in cursor.description]
    return pd.DataFrame(cursor.fetchall(), columns=cols)

run("SELECT DATABASE() AS db, VERSION() AS mysql_version")

,db,mysql_version
0,sakila,8.0.46-google


In [2]:
run("SHOW FULL TABLES")

,Tables_in_sakila,Table_type
0,actor,BASE TABLE
1,actor_info,VIEW
2,address,BASE TABLE
3,category,BASE TABLE
4,city,BASE TABLE
5,country,BASE TABLE
6,customer,BASE TABLE
7,customer_list,VIEW
8,film,BASE TABLE
9,film_actor,BASE TABLE


We first connect to the SQL server where sakila is our acting database. To display all the tables that are in the database, I useed the SHOW command alongside FULL TABLES which gives a list of every type of table in the database. By using FULL we display all the tables whether they are the BASE tables of the views which are queries that someone made from the base tables, saved as its own table. The output is a list of every single table and if we want to (as seen below) we can call all these tables and then filter where the Table_type is a BASE TABLE which is the original raw data.

In [3]:
base = run("SHOW FULL TABLES WHERE Table_type = 'BASE TABLE'")
print(len(base), "base tables")
base

16 base tables


,Tables_in_sakila,Table_type
0,actor,BASE TABLE
1,address,BASE TABLE
2,category,BASE TABLE
3,city,BASE TABLE
4,country,BASE TABLE
5,customer,BASE TABLE
6,film,BASE TABLE
7,film_actor,BASE TABLE
8,film_category,BASE TABLE
9,film_text,BASE TABLE


In [4]:
# How many films are in Sakila 
run("SELECT COUNT(*) AS film_count FROM film")

,film_count
0,1000


To get the film count in the Sakila database we first have to access the table that contains the film information which is the table 'film. We then use SELECT to pull the data accompanied by the COUNT function where we input (*) which just means we are counting all entries from our table of interest which is film. FROM tells the query which table we are pulling the data from which in this case is the 'film' table. I used the AS film_count which acts as a variable where we have now stored the selected counts of all the films from our 'film' table which we can now access by using film_count. 

In [5]:
# Actors whose first name is John
run("""
SELECT actor_id, first_name, last_name
FROM actor
WHERE first_name = 'JOHN'
""")

,actor_id,first_name,last_name
0,192,JOHN,SUVARI


We are now pulling FROM the actor table which contains all the information about the actors. This table has multiple columns, so we have to select our fields of interest which is actor_id, first_name, and last_name, so we know which John we are looking at. To ensure we are only finding people named John, we use the conditional function WHERE which only returns where we only look for first_names that are equal to the entry 'JOHN'. It turns out there is only one actor named John in this databse which is John Suvari. 

In [6]:
# Number of films per category
per_cat = run("""
SELECT c.name AS category, COUNT(fc.film_id) AS film_count
FROM category AS c
JOIN film_category AS fc ON fc.category_id = c.category_id
GROUP BY c.category_id, c.name
ORDER BY film_count DESC, category
""")
print("Total films across categories:", per_cat.film_count.sum())
per_cat

Total films across categories: 1000


,category,film_count
0,Sports,74
1,Foreign,73
2,Family,69
3,Documentary,68
4,Animation,66
5,Action,64
6,New,63
7,Drama,62
8,Games,61
9,Sci-Fi,61


To find the number of films per category we need to pull data from two different tables which are category to get the different types of categories and also film_category which contains the category for each film. This means we need to join the two tables, but since we care about the category we are going to use that as our main table and then join the film category, using category_id as our primary key since it is the two fields that both tables share. Since we have matching field names we need to save each table as its own variable name so we can pull the field using table variable.field. After we join the category_id from our film categories (fc) to our original category (c) table. Since we want the count for each category, we use GROUPBY to aggregate all the counts of each film for each category_id and get the category name (c.name) from our category table. We are then selecting the name of each category from our category table, the count of all the films (originating from film cateogires) which we save as film_count. Then finally to get the largest count first we use ORDERBY DESC to get the film_counts to go from largest to smallest, and we include category so we can see exactly how many films each category has. 

In [7]:
# Customers with more than 40 rentals
run("""
SELECT cu.customer_id, cu.first_name, cu.last_name,
       COUNT(r.rental_id) AS rental_count
FROM customer AS cu
JOIN rental AS r ON r.customer_id = cu.customer_id
GROUP BY cu.customer_id, cu.first_name, cu.last_name
HAVING COUNT(r.rental_id) > 40
""")

,customer_id,first_name,last_name,rental_count
0,75,TAMMY,SANDERS,41
1,144,CLARA,SHAW,42
2,148,ELEANOR,HUNT,46
3,236,MARCIA,DEAN,42
4,526,KARL,SEAL,45


To find customers with more than 40 rentals we need to pull from our customer and rental tables. So that means we need to join the rental tables to our customer table (since we want customers), using customer_id as our primary key to match the tables. We save the customer table as cu and rentals as r so we can match on the identical cusomter_id field name. We then need to aggregate our table using GROUP BY so we can have all the rentals for each customers so we GROUP BY each customer_id, first_name, and last name. We are then interested only in customers with 40+ rentals so we use HAVING to find customers with a rental_ids count that is greater than 40. Then we want to SELECT or return the fields customer_id, first_name, last_name, the count of all the rental_id's that are matched to each customer. 

### Verification counts

In [8]:
# Verification Counts
run("""
SELECT
  (SELECT COUNT(*) FROM information_schema.tables
    WHERE table_schema = 'sakila' AND table_type = 'BASE TABLE') AS base_tables,
  (SELECT COUNT(*) FROM film)                                  AS films,
  (SELECT ROUND(SUM(amount), 2) FROM payment)                  AS payment_total
""")

,base_tables,films,payment_total
0,16,1000,67406.56


The top-5 customers query (with the city of each customer's store) and its two reconciliation checks are in `sql/03_top_customers_by_store_city.sql`. The first query connects customers to their payments using customer_id, then groups those payment rows to calculate each customer's count and total. To find the store's city, it follows the customer's store_id to store, then the store's address_id to address and its city_id to city; using the customer's own address would answer a different question. The check query includes every customer with a LEFT JOIN, treating customers without payments as having a zero total, and compares the sum of all customer totals with the sum taken directly from payment.

In [9]:
cursor.close()
cnx.close()
connector.close()
print("Connection closed")

Connection closed
